# 🛡️ Analyse Exploratoire du Dataset de Malwares
**Projet Machine Learning — 4A Cycle Ingénieur**

Ce notebook couvre :
1. Chargement et aperçu du dataset
2. Distribution des classes
3. Analyse des valeurs manquantes
4. Statistiques descriptives
5. Corrélations entre features
6. Visualisations par classe

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import load_dataset, quick_eda, TARGET_COLUMN

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Chargement du Dataset

In [ ]:
df = load_dataset('../data/dataset.xlsx')
quick_eda(df)

## 2. Distribution des Classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Comptage
counts = df[TARGET_COLUMN].value_counts()
axes[0].bar(counts.index.astype(str), counts.values,
            color=['#43A047', '#E53935'])
axes[0].set_title('Distribution des Classes')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de fichiers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Camembert
axes[1].pie(counts.values, labels=counts.index.astype(str),
            autopct='%1.1f%%', colors=['#43A047', '#E53935'],
            startangle=140)
axes[1].set_title('Proportion Bénin / Malware')
plt.tight_layout()
plt.show()

## 3. Valeurs Manquantes

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print('✅ Aucune valeur manquante détectée !')
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(missing.index, missing.values / len(df) * 100, color='#FF7043')
    ax.set_xlabel('% de valeurs manquantes')
    ax.set_title('Colonnes avec des valeurs manquantes')
    plt.tight_layout()
    plt.show()

## 4. Statistiques Descriptives

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
df_numeric.describe().T.style.background_gradient(cmap='Blues')

## 5. Matrice de Corrélation

In [ ]:
# Top 15 features les plus corrélées à la cible
from sklearn.preprocessing import LabelEncoder
df2 = df.copy()
df2[TARGET_COLUMN] = LabelEncoder().fit_transform(df2[TARGET_COLUMN].astype(str))
df2 = df2.select_dtypes(include=[np.number])

corr_target = df2.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).abs().sort_values(ascending=False)
top_features = corr_target.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df2[top_features + [TARGET_COLUMN]].corr(),
            annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Matrice de Corrélation — Top 15 Features')
plt.tight_layout()
plt.show()

## 6. Distribution des Top Features par Classe

In [ ]:
top6 = corr_target.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, feat in zip(axes.flatten(), top6):
    for label in df[TARGET_COLUMN].unique():
        subset = df[df[TARGET_COLUMN] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.6, label=str(label), density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Distribution des Top Features par Classe', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Entropie — Feature Discriminante Clé
L'entropie des sections est l'une des features les plus discriminantes entre fichiers bénins et malveillants.

In [ ]:
if 'SectionsMeanEntropy' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, color in zip(df[TARGET_COLUMN].unique(), ['#43A047', '#E53935']):
        data = df[df[TARGET_COLUMN] == label]['SectionsMeanEntropy'].dropna()
        ax.hist(data, bins=50, alpha=0.7, color=color,
                label=str(label), density=True)
    ax.axvline(6.5, color='black', linestyle='--', label='Seuil typique (6.5)')
    ax.set_xlabel('Entropie Moyenne des Sections')
    ax.set_ylabel('Densité')
    ax.set_title('Entropie des Sections : Bénin vs Malware')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Colonne SectionsMeanEntropy non disponible dans ce dataset.')